# NASA VEDA — NLDAS-3, CMIP, Black Marble HD

VEDA (NASA's Visualisation, Exploration and Data Analysis platform) is the anonymous,
us-west-2 STAC for **NASA-curated derived products** — climate runs (CMIP6), hydrology
(NLDAS-3), fire/disaster damage, vegetation indices (HLS NDVI), and high-definition Black
Marble nightlights. Every VEDA collection exposes a single `cog_default` data asset (plus a
preview), so the request shape is uniform across the catalogue.

## Endpoint + the flagship picks in the curated catalog (no network)

In [ ]:
from earthlens.stac import Catalog

cat = Catalog()
e = cat.endpoints["veda"]
print(f"url   : {e.url}")
print(f"signer: {e.signer}  region: {e.region}")
veda_keys = sorted(k for k in cat.datasets if k.startswith("veda/"))
for k in veda_keys:
    print(f"  {k}")

## NLDAS-3 over the US Midwest (live)

NLDAS-3 is the North-American Land Data Assimilation System reanalysis — a kilometre-scale,
hourly land-surface state from NASA's Land Information System. Pulling one timestep over a
Midwest bbox illustrates the uniform `cog_default` access pattern.

In [ ]:
import tempfile

from earthlens.core import EarthLens

nldas = EarthLens(
    data_source="veda",
    start="2020-01-01",
    end="2024-12-31",
    dataset="veda/nldas3",
    variables=["cog_default"],
    aoi=[-100.0, 35.0, -90.0, 45.0],
    path=tempfile.mkdtemp(),
    max_items=1,
)
nldas_paths = nldas.download()
nldas_paths

## Black Marble HD nightlights (recipe)

The NASA Black Marble HD nightlights are a disaster-impact product — high-resolution
nighttime-light composites focused on regions affected by disasters. Same request shape.

In [ ]:
bm = EarthLens(
    data_source="veda",
    start="2020-01-01",
    end="2024-12-31",
    dataset="veda/delta-disasters-hd-blackmarble-nightlights",
    variables=["cog_default"],
    aoi=[-78.0, 18.0, -76.0, 19.5],  # Caribbean / disaster-impacted region
    path=tempfile.mkdtemp(),
    max_items=1,
)
bm_paths = bm.download()
bm_paths

## CMIP6 climate-projection slices (recipe)

VEDA mirrors a curated set of CMIP6-derived gridded summaries (the ensemble-median seasonal
anomalies). `CMIP245-winter-median-pr` is the SSP2-4.5 (mid-emissions) winter precipitation
difference; `CMIP585-winter-median-pr` is the SSP5-8.5 (high-emissions) counterpart.

In [ ]:
cmip = EarthLens(
    data_source="veda",
    start="2020-01-01",
    end="2099-12-31",
    dataset="veda/CMIP585-winter-median-pr",
    variables=["cog_default"],
    aoi=[-130.0, 25.0, -65.0, 50.0],  # CONUS
    path=tempfile.mkdtemp(),
    max_items=1,
)
cmip_paths = cmip.download()
cmip_paths

## Inspect

In [ ]:
from pyramids.dataset import Dataset

for label, ps in (
    ("NLDAS-3", nldas_paths),
    ("Black Marble HD", bm_paths),
    ("CMIP6 SSP5-8.5 winter pr", cmip_paths),
):
    ds = Dataset.read_file(ps[0])
    print(
        f"{label:30} bands={ds.band_count} epsg={ds.epsg} shape={ds.read_array().shape}"
    )